Source:
https://www.ncei.noaa.gov/pub/data/ghcn/daily/

Menne, M.J., I. Durre, B. Korzeniewski, S. McNeill, K. Thomas, X. Yin, S. Anthony, R. Ray,
R.S. Vose, B.E.Gleason, and T.G. Houston, 2012: Global Historical Climatology Network -
Daily (GHCN-Daily), Version 3. [indicate subset used following decimal,
e.g. Version 3.12].
NOAA National Climatic Data Center. http://doi.org/10.7289/V5D21VHZ [1/3/26].


In [ ]:
import tarfile
import time
import pandas as pd
import numpy as np

def parse_dly_to_monthly(file_obj):
    # Keys are (year, month)
    # Sums and counts stores separately
    sums = {}
    counts = {}

    for raw in file_obj:
        line = raw.decode("utf-8")
        station = line[0:11]
        year = int(line[11:15])
        month = int(line[15:17])
        element = line[17:21]

        if element not in ("TMAX", "TMIN", "PRCP"):
            continue

        key = (station, year, month)
        if key not in sums:
            sums[key] = {"TMAX": 0, "TMIN": 0, "PRCP": 0}
            counts[key] = {"TMAX": 0, "TMIN": 0, "PRCP": 0}

        # 31 day slots, each 8 chars: VALUE(5) MFLAG(1) QFLAG(1) SFLAG(1)
        for d in range(31):
            i = 21 + d * 8
            v = int(line[i:i+5])
            qflag = line[i+6]

            if v == -9999:
                continue
            if qflag != " ":  # keep only unflagged values
                continue

            sums[key][element] += v
            counts[key][element] += 1

    if not sums:
        return pd.DataFrame(columns=[
            "station_id","year","month","tmax_mean","tmin_mean","prcp_sum","n_tmax","n_tmin","n_prcp"
        ])

    out = []
    for (station, year, month), s in sums.items():
        c = counts[(station, year, month)]
        out.append({
            "station_id": station,
            "year": year,
            "month": month,
            "tmax_mean": (s["TMAX"] / c["TMAX"]) if c["TMAX"] else np.nan,
            "tmin_mean": (s["TMIN"] / c["TMIN"]) if c["TMIN"] else np.nan,
            "prcp_sum":  s["PRCP"] if c["PRCP"] else np.nan,
            "n_tmax": c["TMAX"],
            "n_tmin": c["TMIN"],
            "n_prcp": c["PRCP"],
        })

    return pd.DataFrame(out)

monthly_parts = []
t0 = time.time()
n_files = 0

with tarfile.open("../data/raw/ghcnd_hcn.tar.gz", "r:gz") as tar:
    for member in tar.getmembers():
        if member.isfile() and member.name.endswith(".dly"):
            f = tar.extractfile(member)
            if f is None:
                continue

            df_m = parse_dly_to_monthly(f)
            monthly_parts.append(df_m)

            n_files += 1
            if n_files % 50 == 0:
                print(f"{n_files} station files processed | elapsed {time.time()-t0:.1f}s")

monthly_station = pd.concat(monthly_parts, ignore_index=True)


In [ ]:
monthly_station.head()

In [ ]:
# Converting from tenth units to real
monthly_station["tmax_mean"] = monthly_station["tmax_mean"] / 10.0
monthly_station["tmin_mean"] = monthly_station["tmin_mean"] / 10.0
monthly_station["prcp_sum"]  = monthly_station["prcp_sum"]  / 10.0

# Calculating average temperature
monthly_station["avg_temp"] = (monthly_station["tmax_mean"] + monthly_station["tmin_mean"]) / 2.0
monthly_station.rename(columns={"prcp_sum": "prcp"}, inplace=True)

# Metadata

In [ ]:
# Reading station metadata
stations = pd.read_fwf(
    "../data/raw/ghcnd-stations.txt",
    colspecs=[
        (0, 11),   # station_id
        (12, 20),  # lat
        (21, 30),  # lon
        (31, 37),  # elev
        (38, 40),  # state
    ],
    names=["station_id", "lat", "lon", "elev", "state"],
    dtype={"state": str}
)

# Cleaning up station IDs and names
stations["station_id"] = stations["station_id"].str.strip()
stations["state"] = stations["state"].str.strip()

In [ ]:
# Counting up how many unique states - covers Canada and overseas so more than 50
stations['state'].nunique()

In [ ]:
# Merging df with metadata df
monthly_station = monthly_station.merge(
    stations[["station_id", "state"]],
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Keeping only rows with non-empty state (restricting to North America and overseas terr.)
monthly_station = monthly_station[
    monthly_station['state'].notna()
]

In [ ]:
monthly_station

In [ ]:
# Aggregating by state
state_month = (
    monthly_station
    .groupby(["state", "year", "month"], as_index=False)
    .agg(
        # Averaging temperature over all stations
        avg_temp=("avg_temp", "mean"),
        # Averaging precipitation over all stations
        prcp=("prcp", "mean"),
        # No. stations
        n_stations=("station_id", "nunique")
    )
)

# Only keeping years with quality coverage (theshold set at 3)
state_month = state_month[state_month["n_stations"] >= 3].copy()

In [ ]:
state_month

# Seasonal Demeaning

In [ ]:
state_month["temp_Z"] = (
    state_month["avg_temp"]
    - state_month.groupby(["state", "month"])["avg_temp"].transform("mean")
)

state_month["prcp_Z"] = (
    state_month["prcp"]
    - state_month.groupby(["state", "month"])["prcp"].transform("mean")
)


In [ ]:
state_month

In [ ]:
state_month.groupby("month")["avg_temp"].mean().plot()

# Saving Dataset

In [ ]:
state_month.to_parquet("../data/processed/weather_state_month.parquet", index=False)